# Characterize nodules at their ground-truth locations with MedGemma (all 100 patients)

Runs `5b_characterize_ground_truth_nodules.py` and `5c_characterize_variants.py` (the anchored / few-shot A/B variants) across all 100 downloaded LIDC-IDRI patients, comparing directly against pylidc consensus ground truth in `ground_truth_annotations.json`. Same approach as the 20-patient notebook, just over the full set and pulling in the round-half-up + true top-of-scale (`LIDC-IDRI-0007` #0) few-shot fixes.

Before running anything:
1. **Runtime > Change runtime type > GPU** (T4 is fine, but 100 patients is a much longer run than 20 - consider a faster GPU class if you have one available).
2. Upload **both** `lidc_idri_p1-20.zip` and `lidc_idri_p21-100.zip` to your Google Drive (patients 1-100 + `annotations.csv`, ~1.1GB + ~4.6GB). Skip whichever you already have there from before.
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face access token that has accepted the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

!git clone https://github.com/freya-gul/rail.git
%cd rail
!git checkout medgemma-characterization-variants

Cloning into 'rail'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 168 (delta 81), reused 124 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 74.35 MiB | 1.11 MiB/s, done.
Resolving deltas: 100% (81/81), done.
Updating files: 100% (54/54), done.
Encountered 2 files that should have been pointers, but weren't:
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.pt
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts
/content/rail
error: Your local changes to the following files would be overwritten by checkout:
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.pt
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts
Please commit your changes or stash them before you switch branches.
Aborting


Point these at wherever you uploaded the two zips in Drive. They both unzip into `datasets/LDIC-IDRI-subset/` inside the cloned repo - that's the path these scripts' `DICOM_ROOT` already expects:

In [3]:
ZIP_PATH_1 = "/content/drive/MyDrive/lidc_idri_p1-20.zip"  # <-- update to your actual upload path
ZIP_PATH_2 = "/content/drive/MyDrive/lidc_idri_p21-100.zip"  # <-- update to your actual upload path
DATA_DIR = "datasets/LDIC-IDRI-subset"  # relative to the repo root (we've already %cd'd into rail)

import pathlib
for _p in (ZIP_PATH_1, ZIP_PATH_2):
    assert pathlib.Path(_p).exists(), f"{_p} not found — check the path/upload"

In [4]:
!mkdir -p {DATA_DIR}
!unzip -q {ZIP_PATH_1} -d {DATA_DIR}
!unzip -q {ZIP_PATH_2} -d {DATA_DIR}
!ls {DATA_DIR}/lidc_idri | wc -l

replace datasets/LDIC-IDRI-subset/annotations.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
98


In [5]:
# No monai/simpleitk needed - these scripts never touch the MONAI detector, only
# crop directly out of the raw DICOM series at the ground-truth nodule locations.
!pip install -q pydicom "transformers>=5.12.1" "huggingface_hub>=1.21.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 91.2 MB/s eta 0:00:00


In [6]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

Sanity check: GPU visible to torch (the scripts already default to `cuda` > `mps` > `cpu`):

In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA available: True
NVIDIA A100-SXM4-40GB


## Run baseline characterization (zero-shot, all 100 patients)

Prints live progress per nodule (predicted vs. ground-truth attribute values aren't shown inline, but pass/fail on JSON validation is) plus a running ETA. Resumable at the individual-nodule level - a killed run picks back up partway through a patient rather than redoing it, so it's safe to stop and re-run this cell if Colab disconnects partway through the full 100.

In [8]:
!python image_download/5b_characterize_ground_truth_nodules.py --start 1 --end 100

Loading MedGemma 1.5 on cuda... (288 nodule(s) to characterize)
config.json: 100% 2.55k/2.55k [00:00<00:00, 6.58MB/s]
model.safetensors.index.json: 100% 90.6k/90.6k [00:00<00:00, 27.9MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/8.60G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/8.60G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  13% 1.10G/8.60G [00:03<00:11, 635MB/s,  410MB/s  ]
Reconstructing (incomplete total...):  74% 6.32G/8.60G [00:17<00:05, 437MB/s,  346MB/s  ]

Reconstructing (incomplete total...):  91% 7.86G/8.60G [00:21<00:01, 438MB/s,  437MB/s  ]
Reconstructing (incomplete total...): 100% 8.60G/8.60G [00:22<00:00, 483MB/s,  438MB/s  ]

Fetching 2 files: 100% 2/2 [00:22<00:00, 11.10s/it]
Download complete: 100% 7.79G/7.79G [00:22<00:00, 262MB/s,  262MB/s  ]
Reconstruction complete: 100% 8.60G/8.60G [00:22<00:0

## Peek at partial results anytime

Run this whenever you want - while the run above is still going, if it got interrupted, or once it's fully done. It reads whatever `nodule_characteristics_gt/<patient>.json` files already exist on disk and computes the same MAE/bias summary the final cell below does, over however many nodules have actually been characterized so far. No need to wait for all 100 patients, and safe to re-run repeatedly as more come in.

In [9]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "image_download")
from lidc_attributes import LIDC_ATTRIBUTES

OUTPUT_DIR = Path("image_download/nodule_characteristics_gt")
patient_files = sorted(OUTPUT_DIR.glob("LIDC-IDRI-*.json"))
rows = [row for f in patient_files for row in json.loads(f.read_text())]

print(f"{len(rows)} nodule(s) characterized so far across {len(patient_files)} patient(s)\n")
print(f"{'attribute':<18}{'MAE':>8}{'bias':>8}{'n':>6}   scale")
for a in LIDC_ATTRIBUTES:
    errs = [r[f"{a}_err"] for r in rows if r.get(f"{a}_err") is not None]
    mae = sum(abs(e) for e in errs) / len(errs) if errs else None
    bias = sum(errs) / len(errs) if errs else None
    lo, hi = min(LIDC_ATTRIBUTES[a]["labels"]), max(LIDC_ATTRIBUTES[a]["labels"])
    mae_str = f"{mae:.2f}" if mae is not None else "n/a"
    bias_str = f"{bias:+.2f}" if bias is not None else "n/a"
    print(f"{a:<18}{mae_str:>8}{bias_str:>8}{len(errs):>6}   {lo}-{hi}")

288 nodule(s) characterized so far across 93 patient(s)

attribute              MAE    bias     n   scale
subtlety              1.07   -0.28   276   1-5
internalStructure     0.24   +0.21   276   1-4
calcification         0.31   +0.09   276   1-6
sphericity            0.73   +0.06   276   1-5
margin                0.81   +0.17   276   1-5
lobulation            1.05   +0.12   276   1-5
spiculation           0.71   -0.51   276   1-5
texture               0.72   +0.40   276   1-5
malignancy            0.95   +0.01   276   1-5


## Results

- `image_download/nodule_characteristics_gt/<patient>.json` - per-nodule predicted attributes, ground-truth mean, error, raw MedGemma response, and JSON-validation problems (if any).
- `image_download/characterize_ground_truth_comparison.csv` - the same data flattened across all patients, one row per nodule.
- `image_download/characterize_ground_truth_summary.json` - MAE and signed bias per attribute, aggregated across every nodule.

Quick look at the summary table:

In [10]:
import json
summary = json.load(open("image_download/characterize_ground_truth_summary.json"))
for attr, stats in summary.items():
    print(f"{attr:<18} MAE={stats['mae']:.2f}  bias={stats['bias']:+.2f}  n={stats['n']}")

subtlety           MAE=1.07  bias=-0.28  n=276
internalStructure  MAE=0.24  bias=+0.21  n=276
calcification      MAE=0.31  bias=+0.09  n=276
sphericity         MAE=0.73  bias=+0.06  n=276
margin             MAE=0.81  bias=+0.17  n=276
lobulation         MAE=1.05  bias=+0.12  n=276
spiculation        MAE=0.71  bias=-0.51  n=276
texture            MAE=0.72  bias=+0.40  n=276
malignancy         MAE=0.95  bias=+0.01  n=276


## Try prompt variants: anchored guidance / few-shot examples (all 100 patients)

Runs `5c_characterize_variants.py`, an A/B sibling of the script above:

- `--anchored` - adds targeted anti-underrating guidance to the subtlety/margin/spiculation attribute descriptions. Free (a few extra sentences, no extra images).
- `--fewshot` - prepends two fixed calibration examples (real images + their consensus-rounded ground truth) as prior conversation turns: one unambiguous low-spiculation nodule (`LIDC-IDRI-0005` #0) and one unambiguous high-spiculation nodule (`LIDC-IDRI-0007` #0, spiculation=5.0/4 readers - a true top-of-scale anchor). Costs roughly 2x the vision tokens/time per nodule.

Combinable, and each flag combination writes to its own directory (`nodule_characteristics_gt_anchored/`, `..._fewshot/`, `..._anchored_fewshot/`) so nothing clobbers the zero-shot baseline above.

**Runtime note:** each of these three cells runs over all 100 patients too, and the two `--fewshot` variants roughly double per-nodule time on top of that - this section is easily the most expensive part of the notebook. Feel free to lower `--end` on a cell (e.g. `--end 20`) to spot-check a variant before committing GPU time to the full 100, or just let the baseline above stand on its own if you don't need the A/B comparison this run.

In [11]:
!python image_download/5c_characterize_variants.py --anchored --start 1 --end 100

Variant: anchored=True fewshot=False -> /content/rail/image_download/nodule_characteristics_gt_anchored
Loading MedGemma 1.5 on cuda... (288 nodule(s) to characterize)
Loading weights: 100% 883/883 [00:00<00:00, 5086.29it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py", line 1183, in forward
    model_outputs = self._forward(model_inputs, **forward_params)
               

In [12]:
!python image_download/5c_characterize_variants.py --fewshot --start 1 --end 100

Variant: anchored=False fewshot=True -> /content/rail/image_download/nodule_characteristics_gt_fewshot
Loading MedGemma 1.5 on cuda... (286 nodule(s) to characterize)
Loading weights: 100% 883/883 [00:00<00:00, 4679.88it/s]
Cropping few-shot calibration examples...
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[1/100] LIDC-IDRI-0001 nodule 0 (9.1s): OK  ETA 43m15s (285 nodule(s) left)
[1/100] LIDC-IDRI-0001: 1 nodule(s) -> /content/rail/image_download/nodule_characteristics_gt_fewshot/

In [13]:
!python image_download/5c_characterize_variants.py --anchored --fewshot --start 1 --end 100

Traceback (most recent call last):
  File "/content/rail/image_download/5c_characterize_variants.py", line 28, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2229, in <module>
    from torch import _VF as _VF, functional as functional  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/functional.py", line 8, in <module>
    import torch.nn.functional as F
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/__init__.py", line 8, in <module>
    from torch.nn.modules import *  # usort: skip # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/__init__.py", line 2, in <module>
    from .linear import Bilinear, Identity, LazyLinear, Linear  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py", line

Compare all variants (plus the zero-shot baseline from above) side by side, over whatever patients each has completed so far - safe to re-run any time, doesn't require any of them to be finished:

In [14]:
import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location("variants", "image_download/5c_characterize_variants.py")
variants_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(variants_mod)

variants_mod.compare_variants({
    "baseline": Path("image_download/nodule_characteristics_gt"),
    "anchored": Path("image_download/nodule_characteristics_gt_anchored"),
    "fewshot": Path("image_download/nodule_characteristics_gt_fewshot"),
    "anchored+fewshot": Path("image_download/nodule_characteristics_gt_anchored_fewshot"),
})

KeyboardInterrupt: 

In [17]:
import shutil

source_file = "/content/rail/image_download/nodule_characteristics_gt"
destination_path = "/content/drive/MyDrive/Colab_MedGemma_Results_100/"

# Create the destination directory if it doesn't exist
import os
os.makedirs(destination_path, exist_ok=True)

# Copy the file
for file in os.listdir(source_file):
    shutil.copy(os.path.join(source_file, file), destination_path)

print(f"File copied to: {destination_path}")

File copied to: /content/drive/MyDrive/Colab_MedGemma_Results_100/
